## Create dataset for survival analysis and logistic regression (inc. covariates)
Goal: 
Row = patient
Columns: 

patientuid, 
dob, dob_ordinal, 
date_refus, date_refus_ordinal, 
age_at_refus,
state, 
policy (0 = medical only, 1 = religious exempt, 2 = religious and personal exempt),  party (only Dem, Rep, Bipartisan), 
household_id, 
practiceid, 

state,
county,
rurality, 
neighborhood-level vote share (network effects), 
parent_count (how many potential parents pre-L2 linkage they had)



In [ ]:
import pandas as pd
import zipfile
import ast
import re
import json
import numpy as np
import statistics
import datetime
import matplotlib.pyplot as plt
import gzip
import os
import math

## Load data

In [ ]:
child_politics = pd.read_csv("/share/pi/deho/AFC/mortonc/intermediate/refusals_codes_notes_2018.csv.zip")

In [ ]:
child_politics['dob'] = pd.to_datetime(child_politics['dob'])
child_politics['dob_year'] = child_politics['dob'].dt.year

In [ ]:
child_politics.columns

## Add parent_count

In [ ]:
# parent_count comes create_families.ipynb
parent_count = np.load('/share/pi/deho-pi/AFC/mortonc/intermediate/parent_count.csv.npy', allow_pickle = True)

In [ ]:
parent_count = pd.DataFrame(parent_count)
parent_count.columns = ['patientuid', 'parent_count']

In [ ]:
child_politics = pd.merge(child_politics, parent_count)

In [ ]:
child_politics['party'].value_counts()

In [ ]:
child_politics.columns

In [ ]:
len(child_politics)

## Save

In [ ]:
def save_zip_csv(filepath, dataset):
    # write to CSV
    csv_filename = filepath
    dataset.to_csv(csv_filename, index=False)

    # zip CSV
    zip_filename = csv_filename + '.zip'

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(csv_filename, os.path.basename(csv_filename))

    # remove large csv
    os.remove(csv_filename)
    
    print("Saved!")

In [ ]:
save_zip_csv('/share/pi/deho-pi/AFC/mortonc/processed/child_politics_refusals_covariates_01202026_2018.csv', child_politics)

In [ ]:
sum(child_politics['party'] == 'Republican') + sum(child_politics['party'] == 'Democratic')

In [ ]:
sum((child_politics['party'] == 'Republican') & 
    (child_politics['fips_state'] != 2)) + sum((child_politics['party'] == 'Democratic') & 
                                                     (child_politics['fips_state'] != 2))

In [ ]:
child_politics['household_id'].value_counts()

In [ ]:
child_politics.columns

In [ ]:
sum(child_politics['svi']==-999)